In [1]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="2"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict
from peft import LoraConfig, get_peft_model


import prompts

In [2]:
## prepare data

# read evidence data
evidence_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv')
evidence_ls = evidence_df.loc[:,'text'].dropna().to_list()

#read qq data
qq_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv')
qq_ds = Dataset.from_pandas(qq_df.loc[:,['question','follow_up_questions']])

In [3]:
# load embedding model
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'
model = SentenceTransformer(model_path)

# load model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# calculate embeddings for all data
evidence_embeddings = model.encode(evidence_ls, convert_to_tensor=True).to(device)

In [4]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10]
    # print(top_results)
    res={}
    for idx in top_results:
        tmp=evidence_ls[idx]
        res[tmp]=similarities[idx]
        
    return list(res.keys())

In [5]:
# load generative model
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_gen.resize_token_embeddings(len(tokenizer_gen))

# peft_config = LoraConfig(
#         target_modules=[ "v_proj", "q_proj", "up_proj", "o_proj", "k_proj", "down_proj", "gate_proj" ], 
#         inference_mode=False, 
#         r=4, 
#         lora_alpha=32, 
#         lora_dropout=0.1,    
#         task_type="CAUSAL_LM",        
#     )

# LMmodel = get_peft_model(model_gen, peft_config)

# LMmodel.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding(128256, 4096)

In [37]:
def evaluate_docs(query, docs):
    # print(f"Query : {query}")
    # print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
        
        attention_mask = torch.tensor([[1 for _ in range(len(inputs[0]))]])
        
        # print(len(inputs[0]))
        # print(len(attention_mask[0]))
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
        generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
        
        filter=(generated_text.split('\n')[0])
        # print(filter)
        if '#relevant' in filter:
            outs.append((generated_text.split('\n')[1]).strip())
    
    return outs

In [42]:
def create_new_query_prompt(query, context, fu_question):
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':input},
        {'role':'assistant', 'content':fu_question}
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt").to(device)
    
    return {'inputs': inputs}   
    

In [41]:
qq_train_df = pd.DataFrame(columns=['question', 'relevant_docs', 'follow_up_questions'])
for idx, entry in enumerate(qq_ds):
    if idx == 1: break
    query = entry['question']
    fu_questions = entry['follow_up_questions']
    
    #retriev docs
    docs = retrieve_documents(query)
    
    # evaluate retrieved docs
    rel_docs = evaluate_docs(query, docs)
    
    qq_train_df.loc[len(qq_train_df)]  = [query, '\n'.join(rel_docs), fu_questions]

# make dataset
train_ds = Dataset.from_pandas(qq_train_df)
    
#Apply the tokenization function to the dataset
train_ds = train_ds.map(
    lambda row: create_new_query_prompt(row['question'], row['relevant_docs'], row['follow_up_questions']), 
    batched=False, 
    remove_columns=train_ds.column_names
)

tensor([132,  53,  12, 179,   0, 115, 182,  36, 178, 264], device='cuda:0')
1214
1214
1239
1239
1240
1240
1222
1222
1041
1041
1214
1214
1232
1232
1215
1215
1236
1236
1216
1216


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

TypeError: Provided `function` which is applied to all elements of table returns a variable of type <class 'torch.Tensor'>. Make sure provided `function` returns a variable of type `dict` (or a pyarrow table) to update the dataset or `None` if you are only interested in side effects.

In [13]:
def make_new_query(query, context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=258)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

tensor([122080,    340,    356, 151465,    384, 109975,  49036,    377,    350,
        158352], device='cuda:0')
Query : Who won the ncaa football national championship played in 2016?
----------------------------------------------------------------------------------------------------


/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:452: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


#irrelevant
#irrelevant
#irrelevant
#irrelevant
#relevant
#relevant
#irrelevant
#relevant
#irrelevant
#irrelevant


"['### Which team won the 2016 NCAA Football National Championship in a non-upset fashion?',\n    '### Who defeated Washington in the 2016 NCAA Football National Championship semifinals?',\n    '### Which team won the 2016 NCAA Football National Championship by a margin of 31 points?',\n    '### Which team played against Georgia in the 2016 NCAA Football National Championship and lost in overtime?']"